# Notebook 2: Semantic Similarity Experiments
## SmartCache ML: Sentence Embeddings & Cosine Similarity Calibration

### 1. Problem Overview
Exact-match caching fails when queries differ syntactically but share identical intent. To detect semantic similarity, we map query strings into continuous dense vector representations using a Sentence Transformer model (`all-MiniLM-L6-v2`).

However, semantic matching introduces a fundamental ML trade-off:
- If our similarity threshold is **too low**, we suffer **false positives** (reusing answers for questions that look similar but ask for different things).
- If our similarity threshold is **too high**, we suffer **false negatives** (missing valid paraphrases and paying unnecessary inference cost).

In this notebook, we systematically evaluate similarity thresholds to calibrate our cache.

---
### Learning Objectives
By the end of this notebook, you will be able to:
- Explain dense sentence embeddings and bi-encoder architecture.
- Compute cosine similarity using pure NumPy with $L_2$-normalized dot products.
- Evaluate similarity distributions across positive paraphrase pairs and adversarial negative pairs.
- Calibrate the similarity threshold using Precision, Recall, and F1 curves.
- Defend your threshold choice in an engineering interview.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from smartcache.config import settings
from smartcache.embeddings import encode_query, compute_cosine_similarity, compute_matrix_cosine_similarities

print("Embedding model:", settings.embedding_model_name)

### 2. Computing Embeddings and Cosine Similarity
Let's test our embedding engine on canonical questions, true paraphrases, and adversarial distractors.

In [ ]:
q_canonical = "What is the capital of France?"
q_paraphrase = "Can you tell me the capital city of France?"
q_adversarial = "What is the largest port city in France?"
q_unrelated = "How do you calculate compound interest on an investment?"

vec_can = encode_query(q_canonical)
vec_para = encode_query(q_paraphrase)
vec_adv = encode_query(q_adversarial)
vec_unrelated = encode_query(q_unrelated)

print(f"Embedding dimension: {vec_can.shape[0]}")
print(f"Similarity (Canonical <-> Paraphrase):  {compute_cosine_similarity(vec_can, vec_para):.4f}")
print(f"Similarity (Canonical <-> Adversarial): {compute_cosine_similarity(vec_can, vec_adv):.4f}")
print(f"Similarity (Canonical <-> Unrelated):   {compute_cosine_similarity(vec_can, vec_unrelated):.4f}")

### 3. Measuring Distributions on Positive vs Negative Pairs
Let's load the benchmark dataset, pair queries within the same group (positives) and across adversarial/different groups (negatives), and compute similarity scores.

In [ ]:
df = pd.read_csv(settings.data_raw_path / "benchmark_dataset.csv")

# Collect positive pairs (queries in same group with is_semantic_match=True)
positive_pairs = []
groups = df.groupby('paraphrase_group')
for _, grp in groups:
    queries = grp[grp['is_semantic_match'] == True]['query_text'].tolist()
    if len(queries) >= 2:
        for i in range(len(queries)):
            for j in range(i + 1, min(i + 3, len(queries))):
                positive_pairs.append((queries[i], queries[j]))

# Collect negative adversarial pairs (adversarial queries vs canonicals)
negative_pairs = []
canonicals = df[df['query_role'] == 'canonical']
adversarials = df[df['query_role'] == 'adversarial_negative']

for _, adv_row in adversarials.iterrows():
    # Pair with matching canonical prefix
    base_grp = adv_row['paraphrase_group'].replace('_adv', '')
    matching_can = canonicals[canonicals['paraphrase_group'] == base_grp]
    if not matching_can.empty:
        negative_pairs.append((matching_can.iloc[0]['query_text'], adv_row['query_text']))

# Add some cross-category random negative pairs
unrelated_samples = df.sample(min(40, len(df)), random_state=42)
for i in range(0, len(unrelated_samples) - 1, 2):
    negative_pairs.append((unrelated_samples.iloc[i]['query_text'], unrelated_samples.iloc[i+1]['query_text']))

print(f"Collected {len(positive_pairs)} positive pairs and {len(negative_pairs)} negative pairs.")

In [ ]:
pos_scores = [compute_cosine_similarity(encode_query(p[0]), encode_query(p[1])) for p in positive_pairs]
neg_scores = [compute_cosine_similarity(encode_query(p[0]), encode_query(p[1])) for p in negative_pairs]

plt.figure(figsize=(9, 4))
plt.hist(pos_scores, bins=20, alpha=0.7, label='True Paraphrases (Positives)', color='forestgreen')
plt.hist(neg_scores, bins=20, alpha=0.7, label='Adversarial / Unrelated (Negatives)', color='firebrick')
plt.axvline(0.85, color='black', linestyle='--', label='Default Threshold (0.85)')
plt.title("Cosine Similarity Score Distribution")
plt.xlabel("Cosine Similarity")
plt.ylabel("Pair Count")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 4. Threshold Calibration: Precision, Recall, and F1 Sweep
We evaluate candidate thresholds from 0.70 to 0.95 to find the optimal operating point.

In [ ]:
thresholds = np.linspace(0.65, 0.95, 31)
precisions = []
recalls = []
f1_scores = []

for t in thresholds:
    tp = sum(1 for s in pos_scores if s >= t)
    fn = sum(1 for s in pos_scores if s < t)
    fp = sum(1 for s in neg_scores if s >= t)
    tn = sum(1 for s in neg_scores if s < t)
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 1.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    
    precisions.append(prec)
    recalls.append(rec)
    f1_scores.append(f1)

best_idx = int(np.argmax(f1_scores))
optimal_threshold = thresholds[best_idx]
print(f"Optimal Threshold by F1: {optimal_threshold:.2f}")
print(f"At {optimal_threshold:.2f} -> Precision: {precisions[best_idx]:.3f}, Recall: {recalls[best_idx]:.3f}, F1: {f1_scores[best_idx]:.3f}")

plt.figure(figsize=(8, 4))
plt.plot(thresholds, precisions, label='Precision (Low False Hits)', color='blue')
plt.plot(thresholds, recalls, label='Recall (High Cache Hit Rate)', color='orange')
plt.plot(thresholds, f1_scores, label='F1 Score', color='green', linewidth=2)
plt.axvline(optimal_threshold, color='red', linestyle=':', label=f'Optimal: {optimal_threshold:.2f}')
plt.xlabel("Cosine Similarity Threshold")
plt.ylabel("Score")
plt.title("Threshold Calibration Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 5. False Positive & False Negative Edge Cases
Let's inspect concrete failure cases:
1. **False Positive**: Two queries score above threshold but have different real-world meaning.
2. **False Negative**: Two queries have identical intent but score below threshold due to vocabulary divergence.

In [ ]:
# Inspect adversarial false positives at relaxed threshold (e.g. 0.75)
print("=== ADVERSARIAL PAIRS SCORING ABOVE 0.75 ===")
for p in negative_pairs[:10]:
    sim = compute_cosine_similarity(encode_query(p[0]), encode_query(p[1]))
    if sim >= 0.75:
        print(f"Score: {sim:.3f}\n  Q1: {p[0]}\n  Q2: {p[1]}\n")

### 6. Interview Concepts Checklist
1. **Bi-encoder vs Cross-encoder**: Why bi-encoders allow precomputing vectors for $O(1)$/$O(N)$ dot product search, while cross-encoders require full forward-pass concatenation ($O(N \times \text{model})$).
2. **Precision-Recall Trade-off in Caching**: A false cache hit serves erroneous information to the user (severe quality bug), whereas a false cache miss merely costs minor compute/latency. Therefore, caching favors high precision (e.g. threshold 0.85).
3. **L2 Normalization & Dot Product**: Why normalizing vectors to unit length transforms cosine similarity into a single hardware-accelerated matrix multiplication.